# 02_recorded data session

End-to-end notebook for recording EEG during the n-back cognitive task with a (simulated or real) BrainAccess device. Parameters, device scan/selection, streaming/recording, and data saving are all explicit.

In [1]:
from pathlib import Path
import json
import random

import numpy as np
import matplotlib.pyplot as plt
from sympy import false

from hotb_2026.brainaccess_connection import (
    DEVICE_NAME,
    MONTAGE,
    HARDWARE_SFREQ,
    list_devices,
    BrainAccessDriver,
    SimulatedDriver,
    Electrode,
    Montage,
)
from hotb_2026.record import run_session, run_experiment


## 1. Define montage and other EEG params

Single source of truth for the montage, device parameters, and cognitive-task parameters. Set `SIMULATE=True` to run without hardware.

In [5]:
# ---- mode ----
SIMULATE = False  # set False to use a real BrainAccess headset

# ---- EEG / device params ----
sfreq = 250.0            # hardware: 250 or 500; simulation accepts any positive rate
bias_electrode = 7       # 0-based electrode index, or None to disable feedback
adapter_index = None     # Linux only: hci<N> index, or None for default
chunk_size = 25          # BLE chunk size (1-100)
gain = 8                 # amplifier gain

# Build montage explicitly (edit electrodes here if your cap wiring differs).
from dataclasses import replace
montage = replace(MONTAGE, sfreq=sfreq, bias_electrode=bias_electrode)

if not SIMULATE and round(montage.sfreq) not in HARDWARE_SFREQ:
    raise ValueError(
        f"BrainAccess hardware supports {HARDWARE_SFREQ}, not {montage.sfreq:g} Hz. "
        "Use SIMULATE=True or set sfreq to 250/500."
    )

# ---- cognitive-task params ----
subject_id = "S001"
session_idx = 1
n_sessions = 1
seed = 42
fullscreen = False
min_duration = 5   # seconds per block (use 30+ for real sessions)
max_duration = 8   # seconds per block (use 90 for real sessions)
n_blocks = 3
show_feedback = True

# Data is saved under a folder named after the subject.
out_dir = Path("data") / subject_id
print("Montage:", montage.names, "sfreq=", montage.sfreq, "bias=", montage.bias_electrode)
print("Subject:", subject_id, "out_dir:", out_dir)


Montage: ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2'] sfreq= 250.0 bias= 7
Subject: S001 out_dir: data/S001


## 2. Scan and connect to the chosen device

`list_devices` blocks for ~5 seconds. Pick the device by index, or override `device_name` manually for renamed headsets.

In [6]:
if SIMULATE:
    device_name = None
    driver = SimulatedDriver(montage, seed=seed if seed is not None else 0)
    print("Using simulated driver")
else:
    devices = list_devices(adapter_index=adapter_index)
    print("Found devices:")
    for i, name in enumerate(devices):
        print(f"  [{i}] {name}")
    if not devices:
        raise RuntimeError("No BrainAccess devices found. Power-cycle the headset and retry.")
    DEVICE_INDEX = 0
    device_name = devices[DEVICE_INDEX]
    # device_name = "BA MINI 078"  # manual override if needed

    driver = BrainAccessDriver(
        device_name,
        montage,
        adapter_index=adapter_index,
        chunk_size=chunk_size,
        gain=gain,
    )
    print("Selected device:", device_name)

# Note: driver.open()/start() are called by run_session via EEGRecorder.


Found devices:
  [0] BA MINI 068
Selected device: BA MINI 068


## 3. Start stream and cognitive task

`run_session` opens/starts the device stream, runs the PsychoPy n-back procedure (rest / 1-back / 3-back), then stops and writes the session. Set `n_sessions > 1` to use `run_experiment`.

The PsychoPy window first shows a **"Connecting to EEG device..."** status screen; the n-back procedure starts only after a successful connection. On failure, a readable error message is shown instead of an abrupt crash.

In [7]:
if n_sessions == 1:
    session_dir = run_session(
        subject_id=subject_id,
        session_idx=session_idx,
        driver=driver,
        montage=montage,
        out_dir=out_dir,
        seed=seed,
        fullscreen=fullscreen,
        min_duration=min_duration,
        max_duration=max_duration,
        n_blocks=n_blocks,
        show_feedback=show_feedback,
    )
    session_dirs = [session_dir]
else:
    session_dirs = run_experiment(
        subject_id=subject_id,
        n_sessions=n_sessions,
        driver=driver,
        montage=montage,
        out_dir=out_dir,
        seed=seed,
        fullscreen=fullscreen,
        min_duration=min_duration,
        max_duration=max_duration,
        n_blocks=n_blocks,
        show_feedback=show_feedback,
    )

print("Saved sessions:")
for d in session_dirs:
    print(" ", d)


[2026-11-15 07:50:47.810] [tid  38368] =============[4.0.0]==============
[2026-11-15 07:50:47.810] [tid  38368] Adapter index: 0
[2026-11-15 07:50:47.810] [tid  38368] Chunk size: 25 Logs enabled ?: 1
[2026-11-15 07:50:47.810] [tid  38368] Update path: ../updates/firmware_HALO.bin 
[2026-11-15 07:50:47.810] [tid  38368] Log : path: bacore.log
[2026-11-15 07:50:47.810] [tid  38368]   buffer size: 512 autoflush ?: 0
[2026-11-15 07:50:47.810] [tid  38368]   append ?: 0 level: 2
[2026-11-15 07:50:47.810] [tid  38368]   timestamps enabled ?: 1 thread IDs enabled ?: 1
[2026-11-15 07:50:47.810] [tid  38368] ==================================
[2026-11-15 07:50:47.811] [tid  38368] Connecting to : BA MINI 068
[2026-11-15 07:50:47.811] [tid  38368] Connection attempt 0
[2026-11-15 07:50:53.572] [tid  38368] Could not establish Bluetooth connection!
[2026-11-15 07:50:53.572] [tid  38368] Connection attempt 0 FAILED!
[2026-11-15 07:50:53.572] [tid  38368] Connection attempt 1
[2026-11-15 07:50:59

DeviceError: Could not connect to 'BA MINI 068': Connection error; check that the headset is on and in range

## 4. Save data from notebook with the given subject name

`run_session` already wrote `data/<subject_id>/session_<n>/`. Load and verify the outputs, then plot.

In [ ]:
import json

for session_dir in session_dirs:
    meta = json.loads((session_dir / "meta.json").read_text())
    signal = np.load(session_dir / "signal.npy")
    time = np.load(session_dir / "time.npy")
    changes_path = session_dir / "state_changes.csv"
    print(session_dir)
    print("  subject_id:", meta.get("subject_id"))
    print("  session_id:", meta.get("session_id"))
    print("  electrodes:", meta.get("electrodes"))
    print("  sfreq:", meta.get("sfreq"))
    print("  signal:", signal.shape, signal.dtype)
    print("  time:", time.shape, time.dtype)
    print("  state changes file exists:", changes_path.exists())

# Plot the first channel of the first session with state-change markers.
session_dir = session_dirs[0]
time = np.load(session_dir / "time.npy")
signal = np.load(session_dir / "signal.npy")

changes = np.loadtxt(session_dir / "state_changes.csv", delimiter=",", skiprows=1, dtype=str)
if changes.ndim == 1:
    changes = changes.reshape(1, -1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time, signal[:, 0], lw=0.8)
for row in changes:
    t = float(row[0])
    label = row[1]
    ax.axvline(t, color="r", linestyle="--", alpha=0.6)
    ax.text(t, ax.get_ylim()[1], label, rotation=90, va="top")
ax.set_xlabel("stream time (s)")
ax.set_ylabel("uV")
ax.set_title(f"{subject_id} - {montage.names[0]}")
plt.tight_layout()
plt.show()


### Takeaways

- All EEG/task parameters live in Section 1; change them there and re-run.
- Device scan/selection is isolated in Section 2; set `SIMULATE=True` for offline dev.
- Data is written under `data/<subject_id>/session_<n>/` so subjects never collide.
- Re-running overwrites the same `session_<n>` folder; increment `session_idx` (or timestamp `out_dir`) to keep prior runs.
- On abort (Esc/exception) the current `run_session` stops recording but does not write partial data.